In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
# Make a function that retreives the history
import requests
import csv
from tqdm import tqdm
import html

def get_trial_history(nct_id):
    url = f"https://clinicaltrials.gov/api/int/studies/{nct_id}/history"
    response = requests.get(url)
    if response.status_code == 200:
        try:
            return response.json()
        except Exception as e:
            print(f"Failed to parse JSON data for {nct_id}: {e}")
            return None
    else:
        print(f"Failed to retrieve data for {nct_id}")
        return None

In [ ]:
# Make a function that retreives the most recent record
def get_trial_results(nct_id):
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}?format=json"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to retrieve data for {nct_id}")
        return None

In [ ]:
# Make a function that retreives the version according to version no
import requests
import csv
from tqdm import tqdm
import html

def get_trial_outcomes(nct_id, versionno):
    url = f"https://clinicaltrials.gov/api/int/studies/{nct_id}/history/{versionno}"
    response = requests.get(url)
    if response.status_code == 200:
        try:
            return response.json()
        except Exception as e:
            print(f"Failed to parse JSON data for {nct_id}: {e}")
            return None
    else:
        print(f"Failed to retrieve data for {nct_id}. Status code: {response.status_code}, Reason: {response.reason}")
        return None

In [ ]:
# Download the outcome information for the first version
import os
import json
import pandas as pd

# Read NCT IDs
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()

# Specify the output directory to save downloaded outcome information
output_dir = 'out_pro'
# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Make a function that saves each NCTID's info to a json file
def save_to_json(nct_id, data):
    output_file = os.path.join(output_dir, f"{nct_id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
        
for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):
    
    # Retrieve version numbers
    df=pd.DataFrame(get_trial_history(nct_id)['changes'])
    # Retrieve study start date
    start_date=get_trial_results(nct_id)['protocolSection']['statusModule']['startDateStruct']['date']
    
    # Convert to datetime type 
    df['date'] = pd.to_datetime(df['date'])
    target_date = pd.to_datetime(start_date)
    
    # Find the version before trial starts
    version = df[df['date'] > target_date].iloc[0]['version'] - 1
    
    # Retrieve the right version
    result = get_trial_outcomes(nct_id, version)
    if result:
        try:
            study_protocol_section = result['study']['protocolSection']
            outcomes_data = []
            if 'outcomesModule' in study_protocol_section:
                outcomes_module = study_protocol_section['outcomesModule']
                
                outcome_counters = {'PRIMARY': 0, 'SECONDARY': 0, 'OTHER': 0}
                for outcome_type in ['primaryOutcomes', 'secondaryOutcomes', 'otherOutcomes']:
                    if outcome_type in outcomes_module:
                        outcomes = outcomes_module[outcome_type]
                        for measure in outcomes:
                            priority = measure.get('type', '').upper()
                            if priority not in outcome_counters:
                                priority = outcome_type.replace('Outcomes', '').upper()
                            outcome_counters[priority] += 1
                            outcome_number = outcome_counters[priority]
                            
                            outcome = {
                                'Identifier': f"{priority.capitalize()} {outcome_number}",
                                'Outcome measure': html.unescape(measure.get('measure', 'N/A')),
                                'Outcome description': html.unescape(measure.get('description', 'N/A')),
                                'Time frame': html.unescape(measure.get('timeFrame', 'N/A'))
                            }
                            outcomes_data.append(outcome)

                # Save outcomes data to a JSON file
                output_file = os.path.join(output_dir, f"{nct_id}.json")
                with open(output_file, 'w', encoding='utf-8') as json_file:
                    json.dump(outcomes_data, json_file, ensure_ascii=False, indent=4)
            else:
                print(f"No Outcome Measures Module found for {nct_id}")
        except KeyError as e:
            print(f"Error navigating JSON structure for {nct_id}: {e}")
            error_data = {"error": str(e)}
            output_file = os.path.join(output_dir, f"{nct_id}_error.json")
            with open(output_file, 'w', encoding='utf-8') as json_file:
                json.dump(error_data, json_file, ensure_ascii=False, indent=4)

print("Data extraction complete.")

        

In [ ]:
# Download trial information
import json
import os
from tqdm import tqdm
import requests
import pandas as pd

# Read NCT IDs from the input file
with open('nctid_new150.txt', 'r') as file:
    nct_ids = file.read().splitlines()
    
# Specify the input file path and output directory to save downloaded trial information
output_dir = 'inf_pro'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
    
# Process each NCT ID
for nct_id in tqdm(nct_ids, desc="Processing NCT IDs"):

    # Retrieve version numbers
    df=pd.DataFrame(get_trial_history(nct_id)['changes'])
    # Retrieve study start date
    start_date=get_trial_results(nct_id)['protocolSection']['statusModule']['startDateStruct']['date']
    
    # Convert to datetime type 
    df['date'] = pd.to_datetime(df['date'])
    target_date = pd.to_datetime(start_date)
    
    # Find the version before trial starts
    first_version = df[df['date'] > target_date].iloc[0]['version'] - 1  
    
    result = get_trial_outcomes(nct_id, first_version)
    if result:
        try:
            study_protocol_section = result["study"]['protocolSection']
                
            # Extract relevant information
            trial_info = {
                #'nctid': nct_id,
                #'Start Date': study_protocol_section.get('statusModule', {}).get('startDateStruct', {}).get('date', 'N/A'),
                'Version Before Recruiting': int(first_version), 
                'Most Recent Version': int(df['version'].iloc[-1]),  # Convert to int
                'Official Title': study_protocol_section.get('identificationModule', {}).get('officialTitle', 'N/A'),
                'Brief Summary': study_protocol_section.get('descriptionModule', {}).get('briefSummary', 'N/A'),
                'Eligibility Criteria': study_protocol_section.get('eligibilityModule', {}).get('eligibilityCriteria', 'N/A')
            }
                
            # Save the trial information as a JSON file
            output_file = os.path.join(output_dir, f"{nct_id}.json")
            with open(output_file, 'w', encoding='utf-8') as json_file:
                json.dump(trial_info, json_file, ensure_ascii=False, indent=4)
                
        except KeyError as e:
            print(f"Error navigating JSON structure for {nct_id}: {e}")
            # Save error information as JSON
            error_info = {
                'nctid': nct_id,
                'error': str(e)
            }
            output_file = os.path.join(output_dir, f"{nct_id}_error.json")
            with open(output_file, 'w', encoding='utf-8') as json_file:
                json.dump(error_info, json_file, ensure_ascii=False, indent=4)

print("Data extraction complete.")
